[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Boyu-Zhang-UOI/pml-f2026-notebooks/blob/main/05-deep-learning-pytorch/05_lightning_same_model.ipynb)

# The Same Model, Written Twice

**Session 23 · no homework grades this · read it beside your own training loop**

Session 20 built a training loop by hand, and Session 21 varied it. By now the
loop is familiar enough to be boring — which is the moment a framework starts to
pay. This notebook writes one small model twice, in raw PyTorch and in
Lightning, on the same data with the same seed, and compares what each version
costs you in lines and what it buys.

**On Keras 3:** it is surveyed in the reading this term, not demonstrated and not
assessed. Maintaining three equivalent implementations of one idea costs more
than it teaches, so this notebook keeps two.

In [1]:
import time

import numpy as np
import torch
import lightning as L
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

print("torch", torch.__version__, "| lightning", L.__version__)

digits = load_digits()
X = StandardScaler().fit_transform(digits.data).astype("float32")
y = digits.target.astype("int64")
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.25, random_state=0, stratify=y)

train_ds = TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train))
valid_ds = TensorDataset(torch.from_numpy(X_valid), torch.from_numpy(y_valid))

def loaders(batch_size=64, seed=0):
    gen = torch.Generator().manual_seed(seed)
    return (DataLoader(train_ds, batch_size=batch_size, shuffle=True, generator=gen),
            DataLoader(valid_ds, batch_size=256))

def make_net():
    return nn.Sequential(nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 10))

print("train", X_train.shape, " validation", X_valid.shape)

torch 2.5.1 | lightning 2.6.5
train (1347, 64)  validation (450, 64)


## 1. Raw PyTorch

Every line here is one you have written before. Count them: the parts that are
about *your model* are the two `nn.Linear` calls; everything else is protocol.

In [2]:
EPOCHS = 12

torch.manual_seed(0)
model = make_net()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()
train_loader, valid_loader = loaders(seed=0)

t0 = time.perf_counter()
raw_history = []
for epoch in range(EPOCHS):
    model.train()
    for batch_x, batch_y in train_loader:
        optimizer.zero_grad()
        loss = loss_fn(model(batch_x), batch_y)
        loss.backward()
        optimizer.step()

    model.eval()
    correct = total = 0
    with torch.no_grad():
        for batch_x, batch_y in valid_loader:
            correct += int((model(batch_x).argmax(1) == batch_y).sum())
            total += len(batch_y)
    raw_history.append(correct / total)

raw_seconds = time.perf_counter() - t0
print(f"raw PyTorch  final validation accuracy {raw_history[-1]:.4f}  "
      f"in {raw_seconds:.1f}s")

raw PyTorch  final validation accuracy 0.9378  in 0.2s


## 2. The same thing, in Lightning

The `LightningModule` keeps the model, the step, and the optimizer together; the
`Trainer` owns the loop. Note what disappears: `zero_grad`, `backward`, `step`,
`model.train()`, `model.eval()`, `torch.no_grad()`, and the device handling that
this CPU example did not even need yet.

In [3]:
class DigitClassifier(L.LightningModule):
    def __init__(self, lr=1e-3):
        super().__init__()
        self.save_hyperparameters()
        self.net = make_net()
        self.loss_fn = nn.CrossEntropyLoss()
        self.validation_correct = []

    def forward(self, x):
        return self.net(x)

    def training_step(self, batch, batch_idx):
        x, target = batch
        loss = self.loss_fn(self(x), target)
        self.log("train_loss", loss)
        return loss

    def validation_step(self, batch, batch_idx):
        x, target = batch
        predicted = self(x).argmax(1)
        self.validation_correct.append((predicted == target).float())

    def on_validation_epoch_end(self):
        accuracy = torch.cat(self.validation_correct).mean()
        self.log("val_acc", accuracy, prog_bar=True)
        self.validation_correct.clear()

    def configure_optimizers(self):
        return torch.optim.AdamW(self.parameters(), lr=self.hparams.lr)

In [4]:
L.seed_everything(0, workers=True)
lit_model = DigitClassifier()
trainer = L.Trainer(max_epochs=EPOCHS, accelerator="cpu", logger=False,
                    enable_checkpointing=False, enable_progress_bar=False,
                    enable_model_summary=False, deterministic=True)

train_loader, valid_loader = loaders(seed=0)
t0 = time.perf_counter()
trainer.fit(lit_model, train_loader, valid_loader)
lit_seconds = time.perf_counter() - t0

lit_accuracy = float(trainer.validate(lit_model, valid_loader, verbose=False)[0]["val_acc"])
print(f"\nLightning    final validation accuracy {lit_accuracy:.4f}  in {lit_seconds:.1f}s")

Seed set to 0


GPU available: True (mps), used: False


TPU available: False, using: 0 TPU cores


/Users/boyu/miniconda3/envs/ml-cv-apple/lib/python3.12/site-packages/lightning/pytorch/trainer/setup.py:175: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


/Users/boyu/miniconda3/envs/ml-cv-apple/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=13` in the `DataLoader` to improve performance.
/Users/boyu/miniconda3/envs/ml-cv-apple/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=13` in the `DataLoader` to improve performance.


`Trainer.fit` stopped: `max_epochs=12` reached.



Lightning    final validation accuracy 0.9378  in 0.2s


## 3. Are they the same model?

In [5]:
print(f"{'implementation':<16} {'val accuracy':>13} {'seconds':>9} {'parameters':>12}")
print(f"{'raw PyTorch':<16} {raw_history[-1]:>13.4f} {raw_seconds:>9.1f} "
      f"{sum(p.numel() for p in model.parameters()):>12,}")
print(f"{'Lightning':<16} {lit_accuracy:>13.4f} {lit_seconds:>9.1f} "
      f"{sum(p.numel() for p in lit_model.parameters()):>12,}")
gap = abs(raw_history[-1] - lit_accuracy)
print(f"\ndifference in accuracy: {gap:.4f}"
      f"{'  (identical)' if gap == 0 else ''}")
print("Same architecture, same optimizer, same seed. Any difference would come from")
print("shuffling order or float non-determinism — never from the framework itself.")

implementation    val accuracy   seconds   parameters
raw PyTorch             0.9378       0.2        4,810
Lightning               0.9378       0.2        4,810

difference in accuracy: 0.0000
Same architecture, same optimizer, same seed. Any difference would come from
shuffling order or float non-determinism — never from the framework itself.


## 4. What Lightning actually removed

In [6]:
removed = [
    "optimizer.zero_grad()",
    "loss.backward()",
    "optimizer.step()",
    "model.train() / model.eval()",
    "torch.no_grad() around evaluation",
    "the epoch loop and the batch loop",
    "manual accuracy accumulation across batches",
    "device placement (.to(device)) for model and every batch",
]
for line in removed:
    print("  -", line)
print(f"\n{len(removed)} pieces of protocol, none of them about this model.")

  - optimizer.zero_grad()
  - loss.backward()
  - optimizer.step()
  - model.train() / model.eval()
  - torch.no_grad() around evaluation
  - the epoch loop and the batch loop
  - manual accuracy accumulation across batches
  - device placement (.to(device)) for model and every batch

8 pieces of protocol, none of them about this model.


Two of those matter more than the rest:

- **`model.eval()` and `no_grad()`** are the ones people forget, and forgetting
  them is silent (Session 21 measured the cost).
- **Device placement** is invisible here because this is CPU-only. On a GPU it
  is the line that breaks a script when someone moves it to a different machine,
  and Lightning removes it entirely.

What you give up is directness: when something goes wrong inside `trainer.fit`,
you are debugging someone else's loop. That is a real cost, and it is why this
course teaches the loop first.

## 5. The features that are the actual argument

The boilerplate is a small win. Checkpointing, early stopping and multi-device
training are the reason people adopt Lightning — declared as callbacks rather
than written.

In [7]:
from pathlib import Path

from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint

L.seed_everything(0, workers=True)
lit2 = DigitClassifier()
checkpoint = ModelCheckpoint(monitor="val_acc", mode="max", save_top_k=1,
                             dirpath="lightning_demo", filename="best")
stopper = EarlyStopping(monitor="val_acc", mode="max", patience=4)

trainer2 = L.Trainer(max_epochs=60, accelerator="cpu", logger=False,
                     callbacks=[checkpoint, stopper], enable_progress_bar=False,
                     enable_model_summary=False, deterministic=True)
train_loader, valid_loader = loaders(seed=0)
trainer2.fit(lit2, train_loader, valid_loader)

print(f"\nasked for 60 epochs, stopped after {trainer2.current_epoch}")
print(f"best validation accuracy: {float(checkpoint.best_model_score):.4f}")
print(f"checkpoint written to    : {Path(checkpoint.best_model_path).name}")

Seed set to 0


GPU available: True (mps), used: False


TPU available: False, using: 0 TPU cores


💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.



asked for 60 epochs, stopped after 28
best validation accuracy: 0.9756
checkpoint written to    : best.ckpt


In [8]:
import shutil
shutil.rmtree("lightning_demo", ignore_errors=True)
print("demo checkpoint directory removed")

demo checkpoint directory removed


That is eight lines for behaviour that took a page of bookkeeping in Session 21's
harness — and, importantly, the same eight lines whether you are on one CPU or
eight GPUs.

## What to take from this

- Lightning does not change the model, the optimizer or the result. It removes
  the protocol around them.
- What it removes is exactly the part that is easy to get silently wrong:
  `eval()` mode, `no_grad()`, device placement.
- The cost is indirection when something breaks — which is why you learn the
  loop first and adopt the framework second.
- Checkpointing and early stopping as callbacks are the real argument, not the
  line count.
- Keras 3 is this term's reading-only survey: it is a third way to write the
  same thing, and it is not assessed.

## Where to go next

- **Reading, Session 23** — transfer learning, Lightning, and the Keras 3
  survey.
- **`02_pytorch_cnn.ipynb`** — the ResNet transfer sections this session runs in
  class.
- **HW 4** — raw PyTorch, deliberately: the loop is still the thing being
  assessed.